# Anggota 2: YOLOv8s baseline dan ablasi resolusi masukan (`imgsz`)

Tugas besar Deep Learning, T1-B (platelet sebagai objek kecil, TXL-PBC). Model: YOLOv8s berbobot pralatih COCO.
Pelatihan dan pemilihan hanya memakai `train_clean` (787) dan `val_clean` (239). **`test` (126) dan `test_unflagged` (107) tidak disentuh di notebook ini.**

**Cara pakai**
- Jalankan sel dari atas ke bawah pada runtime GPU (T4). Bagian 5 melatih model; tiap sel run **melewati run yang sudah ada di `logs/runs_anggota2.csv`**, jadi tidak akan melatih ulang apa yang sudah tercatat.
- Konfigurasi dan metrik tiap run ada di `logs/runs_anggota2.csv` dan `configs/yolo/<run>/` (`args.yaml`, `results.csv`). Bobot (`best.pt`) disimpan di Google Drive dan tidak ada di repo.
- Sel di bagian **Arsip** (paling bawah) adalah catatan sesi awal dan sengaja dikunci dengan `raise`. Jangan dijalankan ulang.
- Output sel yang dipertahankan adalah output asli dari sesi Colab; tidak ada yang diedit.

**Faktor yang dikunci untuk semua run:** `yolov8s.pt`, 100 epoch, `patience=100` (tanpa early stopping), `batch=16`, `workers=2`, `deterministic=True`, `optimizer=auto` (hasilnya AdamW dengan lr yang sama di semua run), augmentasi bawaan Ultralytics. Yang berubah hanya `imgsz` dan `seed`.

## 0. Setup sesi (output dari sesi pertama)
Klon repo, pasang paket, unduh dataset pada commit yang ditentukan, validasi (harus 1.260 citra dan 18.143 kotak), lalu buat konfigurasi YOLO. `make_yolo_config.py` wajib dijalankan di setiap mesin karena YAML berisi path absolut.

In [ ]:
!nvidia-smi
!git clone https://github.com/a-mint505/tugas_besar_deep_learning_a_kelompok_deep_cheating
%cd tugas_besar_deep_learning_a_kelompok_deep_cheating
!pip install -q ultralytics pyyaml pycocotools
!git clone https://github.com/lugan113/TXL-PBC_Dataset data/raw/txl_pbc
!git -C data/raw/txl_pbc checkout fd9aadd9636d590ce5b25a1936e61bff4e581a48
!python scripts/validate_dataset.py data/raw/txl_pbc

Mon Nov 16 04:34:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!python scripts/make_yolo_config.py
!ls -l data/processed/yolo/

train_clean: 787 citra -> /content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/train_clean.txt
val_clean: 239 citra -> /content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/val_clean.txt
test: 126 citra -> /content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/test.txt
test_unflagged: 107 citra -> /content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/test_unflagged.txt
yaml: /content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/txl_pbc_clean.yaml
yaml: /content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/txl_pbc_clean_test_unflagged.yaml
kelas: {0: 'WBC', 1: 'RBC', 2: 'Platelets'}
total 180
-rw-r--r-- 1 root root  16884 Nov 16 04:34 test.txt
-rw-r--r-- 1 root root  14338 Nov 16 04:34 test_unflagged.txt
-rw-r--r-- 1 root root 106245 Nov 16 04:34 train_clean.txt
-rw-r--r-- 1 root root    347 Nov 16 04:34 txl_pbc_clean_test_unflagged.

## 1. Lingkungan, default Ultralytics, dan gerbang data
Mencetak versi lingkungan (dicatat ke kolom `env` di log), nilai default argumen pada versi terpasang, lalu memeriksa jumlah split (787/239/126), keberadaan semua citra dan label, nama kelas, dan tidak adanya tumpang tindih antar split.

In [ ]:
import sys, platform, subprocess, yaml, torch, torchvision, ultralytics
from pathlib import Path
sys.path.insert(0, ".")
from ultralytics import YOLO
from ultralytics.cfg import DEFAULT_CFG

ENV = (f"python {platform.python_version()}; torch {torch.__version__}; "
       f"torchvision {torchvision.__version__}; ultralytics {ultralytics.__version__}; "
       f"gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(ENV)
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()

# Verifikasi nama argumen + nilai default di versi TERPASANG
for k in ["seed","deterministic","epochs","patience","batch","imgsz","optimizer","lr0","lrf",
          "close_mosaic","mosaic","scale","translate","fliplr","flipud","degrees","mixup",
          "copy_paste","hsv_h","hsv_s","hsv_v","amp","workers","cos_lr","rect","conf","iou","max_det","split"]:
    print(f"{k:14s}", getattr(DEFAULT_CFG, k, "<TIDAK ADA di versi ini>"))

# Gerbang data (YAML tanpa kunci 'path': train/val/test = path absolut ke file .txt)
CFG_YAML = Path("data/processed/yolo/txl_pbc_clean.yaml").resolve()
d = yaml.safe_load(CFG_YAML.read_text()); print(d)

def read_list(entry):
    p = Path(entry)
    assert p.exists(), f"file daftar tidak ada: {p} -> jalankan make_yolo_config.py di sesi ini"
    return [Path(x) for x in p.read_text().split()]

lists = {k: read_list(d[k]) for k in ("train", "val", "test")}
counts = {k: len(v) for k, v in lists.items()}
print(counts)
assert counts["train"] == 787 and counts["val"] == 239 and counts["test"] == 126, "Jumlah tidak sesuai: STOP, kabari ketua"

# Pastikan citra dan labelnya benar-benar ada (periksa semua, bukan sampel)
for k, v in lists.items():
    miss_img = [p for p in v if not p.exists()]
    miss_lab = [p for p in v if not Path(str(p).replace("/images/", "/labels/")).with_suffix(".txt").exists()]
    print(k, "citra hilang:", len(miss_img), "| label hilang:", len(miss_lab))
    assert not miss_img, miss_img[:3]
    assert not miss_lab, miss_lab[:3]

assert d["names"] in ({0: "WBC", 1: "RBC", 2: "Platelets"}, ["WBC", "RBC", "Platelets"]), d["names"]

# Tidak boleh ada tumpang tindih antar split
assert not (set(lists["train"]) & set(lists["val"])) and not (set(lists["train"]) & set(lists["test"])) \
       and not (set(lists["val"]) & set(lists["test"])), "Ada citra di lebih dari satu split!"
print("Gerbang data OK")

python 3.13.16; torch 2.14.1+cu130; torchvision 0.29.1+cu130; ultralytics 8.4.215; gpu Tesla T4
seed           0
deterministic  True
epochs         100
patience       100
batch          16
imgsz          640
optimizer      auto
lr0            0.01
lrf            0.01
close_mosaic   10
mosaic         1.0
scale          0.5
translate      0.1
fliplr         0.5
flipud         0.0
degrees        0.0
mixup          0.0
copy_paste     0.0
hsv_h          0.015
hsv_s          0.7
hsv_v          0.4
amp            True
workers        8
cos_lr         False
rect           False
conf           None
iou            None
max_det        300
split          val
{'train': '/content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/train_clean.txt', 'val': '/content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/val_clean.txt', 'test': '/content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/test.txt', 'names': {0: 'WBC', 1: 'RBC', 2: 

## 2. Google Drive dan `log_run`
Run ditulis langsung ke Drive agar bobot selamat saat sesi putus. Signature `log_run` adalah `(member, **row)` dan kolom log ada di `RUN_FIELDS`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/tubes_dl/anggota2")
(DRIVE / "logs").mkdir(parents=True, exist_ok=True)
RUNS_DIR = str(DRIVE / "runs")

import inspect
from scripts.utils import log_run, RUN_FIELDS
print("SIGNATURE:", inspect.signature(log_run))
print("RUN_FIELDS:", RUN_FIELDS)

Mounted at /content/drive
SIGNATURE: (member: str, **row)
RUN_FIELDS: ['run_id', 'date', 'member', 'model', 'purpose', 'ablation_factor', 'factor_value', 'seed', 'train_split', 'val_split', 'imgsz_or_min_size', 'epochs', 'batch_size', 'lr', 'pretrained', 'env', 'config_path', 'commit', 'val_map50', 'val_map50_95', 'val_ap50_platelet', 'val_mae_platelet', 'params_m', 'latency_ms', 'status', 'notes']


## 3. Konfigurasi
Sel ini berhenti jika log belum ada. Itu disengaja: tanpa log, sel run mengira belum ada run yang selesai dan akan melatih ulang. Ambil log dengan `git pull`, atau salin dari `MyDrive/tubes_dl/anggota2/logs/runs_anggota2.csv`.

In [ ]:
import time, traceback, shutil
import pandas as pd
from pathlib import Path

MEMBER = "anggota2"
LOG = Path(f"logs/runs_{MEMBER}.csv")
assert LOG.exists(), (f"{LOG} tidak ada. Jalankan git pull atau salin dari Drive dulu; "
                      "jangan lanjut, karena run yang sudah selesai akan dilatih ulang.")
print(len(pd.read_csv(LOG)), "baris di log")

## 4. Definisi `run_yolo`
Satu run = latih 100 epoch, validasi `best.pt` pada `val_clean` (`split="val"`), lalu catat ke log (termasuk run yang gagal, dengan pesan galat di `notes`). Penjaga di awal menolak folder run yang sudah ada, supaya Ultralytics tidak diam-diam membuat folder bernama lain. Platelet AP50-95, presisi, dan recall disimpan di kolom `notes` karena tidak ada kolom khusus di `RUN_FIELDS`; `val_mae_platelet` dan `latency_ms` dikosongkan (dihitung skrip evaluasi bersama).

In [ ]:
import time, traceback, shutil
RUNS_DIR = str(DRIVE / "runs")
EPOCHS, BATCH, WORKERS = 100, 16, 2          # = args.yaml baseline

def run_yolo(imgsz, seed, name, purpose, ablation_factor="", factor_value="", note=""):
    assert not (Path(RUNS_DIR) / name).exists(), f"folder sudah ada: {name} -> putuskan manual (hapus / resume)"
    t0 = time.time()
    row = {f: None for f in RUN_FIELDS if f != "member"}
    row.update(run_id=name, date=time.strftime("%Y-%m-%d"), model="yolov8s", purpose=purpose,
               ablation_factor=ablation_factor, factor_value=factor_value, seed=seed,
               train_split="train_clean", val_split="val_clean", imgsz_or_min_size=imgsz,
               epochs=EPOCHS, batch_size=BATCH, pretrained="COCO (yolov8s.pt)", env=ENV,
               config_path=str(CFG_YAML.relative_to(Path.cwd())), commit=GIT_COMMIT, status="failed")
    notes = [note] if note else []
    try:
        model = YOLO("yolov8s.pt")
        model.train(data=str(CFG_YAML), imgsz=imgsz, epochs=EPOCHS, patience=EPOCHS, batch=BATCH,
                    seed=seed, deterministic=True, workers=WORKERS, device=0,
                    project=RUNS_DIR, name=name, exist_ok=False, plots=True)
        save_dir = Path(model.trainer.save_dir)
        best = save_dir / "weights" / "best.pt"; assert best.exists(), best
        res = pd.read_csv(save_dir / "results.csv"); res.columns = res.columns.str.strip()
        assert len(res) == EPOCHS, f"results.csv {len(res)} baris != {EPOCHS}"
        row["lr"] = float(res["lr/pg0"].max())
        notes.append("lr = puncak lr/pg0 di results.csv (hasil optimizer=auto)")
        if "time" in res.columns: notes.append(f"waktu_train_s={res['time'].iloc[-1]:.0f}")
        ym = YOLO(str(best))
        m = ym.val(data=str(CFG_YAML), split="val", imgsz=imgsz, batch=BATCH, device=0,
                   plots=False, project=RUNS_DIR, name=name + "_val", exist_ok=True)
        b = m.box
        assert {int(k): str(v) for k, v in m.names.items()} == {0: "WBC", 1: "RBC", 2: "Platelets"}
        pc = {int(c): dict(ap50=float(b.ap50[i]), ap=float(b.ap[i]), p=float(b.p[i]), r=float(b.r[i]))
              for i, c in enumerate(b.ap_class_index)}
        assert 2 in pc, "Platelets tidak ada di hasil val"
        row.update(val_map50=float(b.map50), val_map50_95=float(b.map), val_ap50_platelet=pc[2]["ap50"],
                   params_m=round(sum(p.numel() for p in ym.model.parameters()) / 1e6, 2), status="ok")
        notes += [f"platelet AP50-95={pc[2]['ap']:.4f} P={pc[2]['p']:.4f} R={pc[2]['r']:.4f}",
                  f"WBC AP50-95={pc.get(0,{}).get('ap')} RBC AP50-95={pc.get(1,{}).get('ap')}",
                  f"best={best}", "val_mae_platelet & latency_ms: dari skrip evaluasi bersama"]
    except Exception as e:
        notes.append(f"GAGAL: {type(e).__name__}: {e}"); traceback.print_exc()
    finally:
        notes.append(f"waktu_total_s={round(time.time()-t0)}")
        row["notes"] = " | ".join(notes)
        log_run(member=MEMBER, **row)
    return row


## 5. Menjalankan run
Urutan eksekusi pada sesi asli (setiap sel melewati run yang sudah tercatat):

1. **160 dan 224, seed 42** (sel di bawah, `PLAN` memuat seluruh rencana awal, `RUN_NOW` hanya dua run pertama).
2. **640 seed 2024, 320 seed 42, 480 seed 42.**
3. **480 dan 320, seed 123 dan 2024.**

Run 640 seed 42 (baseline) dan 640 seed 123 dijalankan lebih awal; catatannya ada di bagian Arsip. Run 640 seed 123 pernah terputus di epoch 3 saat runtime ter-reset, lalu dilatih ulang dari awal; folder yang terputus dipindah ke `_interrupted_y8s_img640_s123` di Drive dan tidak dipakai.

In [ ]:
PLAN = [(160, 42), (224, 42), (320, 42), (480, 42), (640, 123), (640, 2024)]
RUN_NOW = PLAN[:2]          # mulai dari 2 run; perluas setelah tahu waktunya

done = LOG.read_text() if LOG.exists() else ""
for imgsz, seed in RUN_NOW:
    name = f"y8s_img{imgsz}_s{seed}"
    if name in done: print("lewati (sudah di log):", name); continue
    r = run_yolo(imgsz, seed, name, purpose="ablation" if seed == 42 else "seed_variance",
                 ablation_factor="imgsz", factor_value=imgsz)
    shutil.copy(LOG, DRIVE / "logs" / LOG.name)
    print({k: r[k] for k in ("run_id", "status", "val_map50", "val_map50_95", "val_ap50_platelet")})
    if r["status"] != "ok": break

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/txl_pbc_clean.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=160, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scal

In [ ]:
assert "run_yolo" in globals(), "run_yolo belum didefinisi: tempel sel lengkap run_yolo"
RUN_NOW = [(640, 2024), (320, 42), (480, 42)]

done = LOG.read_text()
for imgsz, seed in RUN_NOW:
    name = f"y8s_img{imgsz}_s{seed}"
    if name in done: print("lewati (sudah di log):", name); continue
    r = run_yolo(imgsz, seed, name,
                 purpose="ablation" if seed == 42 else "seed_variance",
                 ablation_factor="imgsz", factor_value=imgsz)
    shutil.copy(LOG, DRIVE / "logs" / LOG.name)
    print({k: r[k] for k in ("run_id", "status", "val_map50", "val_map50_95", "val_ap50_platelet")})
    if r["status"] != "ok": break

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/txl_pbc_clean.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scal

In [ ]:
import shutil
RUN_NOW = [(480, 123), (480, 2024), (320, 123), (320, 2024)]

done = LOG.read_text()
for imgsz, seed in RUN_NOW:
    name = f"y8s_img{imgsz}_s{seed}"
    if name in done: print("lewati (sudah di log):", name); continue
    r = run_yolo(imgsz, seed, name, purpose="seed_variance",
                 ablation_factor="imgsz", factor_value=imgsz)
    shutil.copy(LOG, DRIVE / "logs" / LOG.name)
    print({k: r[k] for k in ("run_id", "status", "val_map50", "val_map50_95", "val_ap50_platelet")})
    if r["status"] != "ok": break

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/txl_pbc_clean.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scal

## 6. Pemeriksaan
**Catatan:** output sel pertama di bawah diambil sebelum baris baseline di log dikoreksi, sehingga `NOTES BASELINE` masih menampilkan teks sementara dan tabel perbandingannya baru memuat tiga run. Keadaan terkini ada di bagian 7.

Yang dibuktikan sel ini: (a) puncak `lr/pg0` identik (0.001391) di run 160, 224, dan 640, jadi `optimizer=auto` memilih setelan yang sama di semua level dan hanya resolusi yang berubah; (b) ada baris label yang persis sama (duplikat) di semua split: train 19, val 15, test 13, terpusat pada sedikit citra.

In [ ]:
import pandas as pd
from pathlib import Path

RUNS = DRIVE / "runs"          # semua run sekarang di Drive
LOG = Path("logs/runs_anggota2.csv")

# 1) Optimizer/lr tiap run, dibaca dari results.csv (kebenaran dasar)
print("== lr/pg0 per run (baseline harus mirip 160 dan 224) ==")
for rd in sorted(RUNS.glob("y8s_img*_s*")):
    if rd.name.endswith("_val"): continue
    r = pd.read_csv(rd / "results.csv"); r.columns = r.columns.str.strip()
    print(f"{rd.name:20s} epoch={len(r):3d}  lr_maks={r['lr/pg0'].max():.6f}  waktu_s={r['time'].iloc[-1]:.0f}")

# 2) Notes baseline (cek placeholder)
df = pd.read_csv(LOG)
print("\n== NOTES BASELINE ==")
print(df.loc[df.run_id == "y8s_img640_s42", "notes"].iloc[0])

# 3) Duplikat label per split
def dup_stats(paths):
    tot = dup = 0; per = {}
    for p in paths:
        lab = Path(str(p).replace("/images/", "/labels/")).with_suffix(".txt")
        rows = [tuple(l.split()) for l in lab.read_text().splitlines() if l.strip()]
        n = len(rows) - len(set(rows)); tot += len(rows); dup += n
        if n: per[p.name] = n
    return tot, dup, per
print("\n== DUPLIKAT LABEL (total baris, duplikat, per citra) ==")
for k in ("train", "val", "test"):
    print(k, dup_stats(lists[k]))

# 4) Tabel perbandingan
a = df[df.ablation_factor == "imgsz"].copy()
a["plt_ap50_95"] = a.notes.str.extract(r"platelet AP50-95=([0-9.]+)").astype(float)
a["plt_R"] = a.notes.str.extract(r"R=([0-9.]+)").astype(float)
print("\n== PERBANDINGAN (val_clean, seed 42) ==")
print(a[["run_id", "seed", "imgsz_or_min_size", "val_map50", "val_map50_95",
         "val_ap50_platelet", "plt_ap50_95", "plt_R", "lr", "status"]]
      .sort_values(["imgsz_or_min_size", "seed"]).to_string())

== lr/pg0 per run (baseline harus mirip 160 dan 224) ==
y8s_img160_s42       epoch=100  lr_maks=0.001391  waktu_s=773
y8s_img224_s42       epoch=100  lr_maks=0.001391  waktu_s=836
y8s_img640_s42       epoch=100  lr_maks=0.001391  waktu_s=1506

== NOTES BASELINE ==
baseline yolov8s 640 | optimizer: ISI dari baris 'optimizer:' di log training baseline | platelet AP50-95=0.7922 P=0.9523 R=0.9003 | WBC AP50-95=0.8993907736379188 RBC AP50-95=0.903275511888563 | best=/content/drive/MyDrive/tubes_dl/anggota2/runs/y8s_img640_s42/weights/best.pt | dilatih dgn kode lama (log gagal); val dievaluasi ulang dari best.pt | val_mae_platelet & latency_ms: dari skrip evaluasi bersama

== DUPLIKAT LABEL (total baris, duplikat, per citra) ==
train (11234, 19, {'92129d1652e75c3892beb376df57d274.png': 1, 'a8581e04ac765bd3ad658cf8e530a6ea.png': 1, 'c255521a08b75da09373db44e7f099c5.png': 17})
val (3583, 15, {'0ac132c270355979b686c0801e558849.png': 15})
test (1881, 13, {'1299047edeb15e2581d9680aaa667d5d.png': 

Duplikat pada citra val `0ac132c2…` seluruhnya berkelas `1` (RBC), jadi tidak memengaruhi hitungan platelet pada citra itu. Ultralytics membuang baris duplikat saat memuat label (`15 duplicate labels removed`): val 3.583 → 3.568 instance. Ground truth model lain harus disamakan.

In [ ]:
from collections import Counter
lab = Path(str(next(p for p in lists["val"] if p.name.startswith("0ac132c2"))).replace("/images/", "/labels/")).with_suffix(".txt")
rows = [tuple(l.split()) for l in lab.read_text().splitlines() if l.strip()]
c = Counter(rows); print({k[0]: v for k, v in c.items() if v > 1})   # kelas -> jumlah salinan

{'1': 2}


Koreksi baris baseline di log: kolom `lr` dan catatan optimizer (nilai diambil dari `results.csv`, lalu diverifikasi identik dengan run 160 dan 224). Aman dijalankan ulang; jika sudah dikoreksi, sel hanya mencetak "sudah diperbaiki sebelumnya".

In [ ]:
import pandas as pd
from pathlib import Path
LOG = Path("logs/runs_anggota2.csv")

df = pd.read_csv(LOG)
m = df.run_id == "y8s_img640_s42"
assert m.sum() == 1
old = df.loc[m, "notes"].iloc[0]
if "ISI dari baris" in old:
    df.loc[m, "lr"] = 0.001391
    df.loc[m, "notes"] = old.replace(
        "optimizer: ISI dari baris 'optimizer:' di log training baseline",
        "optimizer: auto; lr_maks lr/pg0=0.001391 dari results.csv, identik dengan run 160 & 224 (koreksi manual setelah verifikasi)")
    df.to_csv(LOG, index=False)
    print("log diperbaiki")
else:
    print("sudah diperbaiki sebelumnya")
print(pd.read_csv(LOG).loc[m, ["run_id", "lr", "notes"]].to_string())

sudah diperbaiki sebelumnya
           run_id        lr                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  notes
1  y8s_img640_s42  0.001391  baseline yolov8s 640 | optimizer: auto; lr_maks lr/pg0=0.001391 dari results.csv, identik dengan run 160 & 224 (koreksi manual setelah verifikasi) | platelet AP50-95=0.7922 P=0.9523 R=0.9003 | WBC AP50-95=0.8993907736379188 RBC AP50-95=0.903275511888563 | best=/content/drive/MyDrive/tubes_dl/anggota2/runs/y8s_img640_s42/weights/best.pt | dilatih dgn kode lama (log gagal); val dievaluasi ulang dari best.pt | val_mae_platelet & latency_ms: da

## 7. Hasil akhir (val_clean)
Tabel per run dan ringkasan per resolusi dari `logs/runs_anggota2.csv`.

In [ ]:
import pandas as pd
df = pd.read_csv("logs/runs_anggota2.csv")
a = df[(df.ablation_factor == "imgsz") & (df.status == "ok")].copy()
a["plt_ap50_95"] = a.notes.str.extract(r"platelet AP50-95=([0-9.]+)").astype(float)
print(a[["run_id", "seed", "imgsz_or_min_size", "val_map50_95", "val_ap50_platelet", "plt_ap50_95", "lr"]]
      .sort_values(["imgsz_or_min_size", "seed"]).to_string())

g = a.groupby("imgsz_or_min_size").agg(n_seed=("seed", "nunique"),
        plt_mean=("plt_ap50_95", "mean"), plt_min=("plt_ap50_95", "min"), plt_max=("plt_ap50_95", "max"),
        mAP5095_mean=("val_map50_95", "mean"), ap50_plt_mean=("val_ap50_platelet", "mean"))
g["rentang"] = g.plt_max - g.plt_min
print(g.round(4).to_string())
print("baris tercatat:", len(df), "| gagal:", (df.status != "ok").sum())

              run_id  seed  imgsz_or_min_size  val_map50_95  val_ap50_platelet  plt_ap50_95        lr
2     y8s_img160_s42    42                160      0.820764           0.938974       0.6680  0.001391
3     y8s_img224_s42    42                224      0.849517           0.954503       0.7548  0.001391
6     y8s_img320_s42    42                320      0.861961           0.975226       0.7745  0.001391
10   y8s_img320_s123   123                320      0.859661           0.958820       0.7748  0.001391
11  y8s_img320_s2024  2024                320      0.862477           0.977661       0.7873  0.001391
7     y8s_img480_s42    42                480      0.867425           0.977616       0.7931  0.001391
8    y8s_img480_s123   123                480      0.863436           0.980139       0.7982  0.001391
9   y8s_img480_s2024  2024                480      0.865092           0.982361       0.7938  0.001391
1     y8s_img640_s42    42                640      0.864961           0.975671    

**Ringkasan platelet AP50-95** (disalin dari tabel di atas):

| imgsz | jumlah seed | rata-rata | min – maks | rentang |
|---|---|---|---|---|
| 160 | 1 | 0,668 | – | belum terukur |
| 224 | 1 | 0,755 | – | belum terukur |
| 320 | 3 | 0,779 | 0,7745 – 0,7873 | 0,0128 |
| 480 | 3 | 0,795 | 0,7931 – 0,7982 | 0,0051 |
| 640 | 3 | 0,794 | 0,7922 – 0,7950 | 0,0028 |

**Batasan yang harus ikut disebut saat melaporkan:**
- Hanya 3 seed per level (160 dan 224 hanya 1), jadi rentang antar-seed adalah gambaran kasar, bukan uji statistik.
- `val_clean` hanya memuat 111 platelet di 63 citra, dan ketidakpastian dari ukuran val tidak tercakup oleh rentang seed.
- Hasil berlaku di bawah augmentasi bawaan Ultralytics (mosaic dll.); ini bukan efek resolusi murni.
- Recall platelet sangat berisik antar-seed (0,865–0,949 pada tiga seed 640) dan tidak boleh dipakai untuk menyimpulkan efek resolusi.
- Ablasi ini menunjukkan pola, bukan penyebab. Dugaan mekanisme (misalnya ukuran piksel platelet pada input) belum diuji di sini.

## 8. Ekspor konfigurasi ke repo
Menyalin `args.yaml` dan `results.csv` tiap run dari Drive ke `configs/yolo/<run>/` (tanpa bobot) dan menautkan kolom `config_path` di log ke berkas itu. Setelah ini, commit hanya `logs/runs_anggota2.csv` dan `configs/yolo` dengan nama eksplisit (jangan `git add .`; `data/`, `*.pt`, dan folder run tidak boleh ikut).

In [ ]:
import shutil, pandas as pd
from pathlib import Path

LOG = Path("logs/runs_anggota2.csv")
df = pd.read_csv(LOG)
print(df[["run_id", "imgsz_or_min_size", "seed", "status", "lr"]].to_string())
assert (df.status == "ok").all(), "ada baris gagal: cek dulu"
assert len(df) == 12, f"harus 12 baris (1 smoke + 11 run), ternyata {len(df)}"

runs = DRIVE / "runs"
for rd in sorted(p for p in runs.iterdir()
                 if p.is_dir() and p.name.startswith("y8s_img") and not p.name.endswith("_val")):
    dst = Path("configs/yolo") / rd.name
    dst.mkdir(parents=True, exist_ok=True)
    for f in ("args.yaml", "results.csv"):
        shutil.copy(rd / f, dst / f)

m = df.run_id.str.startswith("y8s_img")
df.loc[m, "config_path"] = "configs/yolo/" + df.loc[m, "run_id"] + "/args.yaml"
df.to_csv(LOG, index=False)
shutil.copy(LOG, DRIVE / "logs" / LOG.name)
print(sorted(p.name for p in Path("configs/yolo").iterdir()))

              run_id  imgsz_or_min_size  seed status        lr
0      smoke_y8s_160                160    42     ok       NaN
1     y8s_img640_s42                640    42     ok  0.001391
2     y8s_img160_s42                160    42     ok  0.001391
3     y8s_img224_s42                224    42     ok  0.001391
4    y8s_img640_s123                640   123     ok  0.001391
5   y8s_img640_s2024                640  2024     ok  0.001391
6     y8s_img320_s42                320    42     ok  0.001391
7     y8s_img480_s42                480    42     ok  0.001391
8    y8s_img480_s123                480   123     ok  0.001391
9   y8s_img480_s2024                480  2024     ok  0.001391
10   y8s_img320_s123                320   123     ok  0.001391
11  y8s_img320_s2024                320  2024     ok  0.001391
['y8s_img160_s42', 'y8s_img224_s42', 'y8s_img320_s123', 'y8s_img320_s2024', 'y8s_img320_s42', 'y8s_img480_s123', 'y8s_img480_s2024', 'y8s_img480_s42', 'y8s_img640_s123', 'y8s_img640

---
## Arsip: catatan sesi awal (jangan dijalankan ulang)
Empat sel berikut dijalankan pada sesi awal dengan versi pertama `run_yolo` (nama kunci log belum disesuaikan, sehingga pencatatan otomatis gagal). Baseline 640 seed 42 dan run 640 seed 123 dilatih dari sel-sel ini; hasilnya dicatat ulang ke log lewat `eval_and_log`, dan konfigurasi serta kurva latihnya ada di `configs/yolo/`. Sel dikunci dengan `raise` karena mendefinisi ulang `run_yolo` versi lama, dan memicu pelatihan 100 epoch.

1. Definisi `run_yolo` versi awal.
2. Uji asap (1 epoch, 5% data) dan baseline 640 seed 42.
3. `eval_and_log`: validasi ulang `best.pt` baseline pada val dan pencatatan ke log.
4. Run 640 seed 123 (dimulai ulang setelah sesi terputus).

In [ ]:
raise RuntimeError("Sel arsip: sudah dijalankan di sesi awal. Jangan dijalankan ulang (lihat catatan di atas).")  # baris ini ditambahkan saat merapikan notebook; output di bawah adalah catatan asli
import time, traceback, shutil
from scripts.utils import log_run, RUN_FIELDS
print("RUN_FIELDS:", RUN_FIELDS)

EPOCHS = 100          # KEPUTUSAN TIM, bukan dari saya; harus SAMA di semua run ablasi
BATCH, WORKERS = 16, 2
RUNS_DIR = "/content/runs_yolo"   # ganti ke folder Drive (bagian 4)

def log_checked(values):
    """Cocokkan values ke RUN_FIELDS; gagal keras kalau ada yang tak cocok."""
    missing = [k for k in RUN_FIELDS if k not in values]
    extra = [k for k in values if k not in RUN_FIELDS]
    if missing or extra:
        raise KeyError(f"Sesuaikan kunci 'values'. Belum diisi: {missing} | Tidak dikenal: {extra}")
    log_run(**{k: values[k] for k in RUN_FIELDS})   # cek signature log_run di sel ini jika bukan kwargs

def run_yolo(imgsz, seed, name, epochs=EPOCHS, model_name="yolov8s.pt", note="", **extra_train):
    t0 = time.time()
    v = dict(run_name=name, model=model_name, imgsz=imgsz, seed=seed, epochs=epochs, batch=BATCH,
             env=ENV, git_commit=GIT_COMMIT, notes=note, status="failed", error="")
    try:
        model = YOLO(model_name)
        model.train(data=str(CFG_YAML), imgsz=imgsz, epochs=epochs, patience=epochs,
                    batch=BATCH, seed=seed, deterministic=True, workers=WORKERS, device=0,
                    project=RUNS_DIR, name=name, exist_ok=False, plots=True, **extra_train)
        save_dir = Path(model.trainer.save_dir)          # jangan asumsikan path; ambil dari trainer
        best = save_dir / "weights" / "best.pt"; assert best.exists()
        m = YOLO(str(best)).val(data=str(CFG_YAML), split="val", imgsz=imgsz, batch=BATCH,
                                device=0, plots=False, project=RUNS_DIR, name=name + "_val")
        b = m.box
        assert {int(k): str(x) for k, x in m.names.items()} == {0:"WBC",1:"RBC",2:"Platelets"}
        pc = {int(c): dict(ap50=float(b.ap50[i]), ap=float(b.ap[i]), p=float(b.p[i]), r=float(b.r[i]))
              for i, c in enumerate(b.ap_class_index)}
        assert 2 in pc, "Platelets tidak muncul di val: periksa data"
        v.update(status="ok", map50=float(b.map50), map50_95=float(b.map),
                 platelet_ap50=pc[2]["ap50"], platelet_ap=pc[2]["ap"],
                 platelet_p=pc[2]["p"], platelet_r=pc[2]["r"],
                 wbc_ap=pc.get(0,{}).get("ap"), rbc_ap=pc.get(1,{}).get("ap"),
                 best_weights=str(best))
    except Exception as e:
        v["error"] = f"{type(e).__name__}: {e}"; traceback.print_exc()
    finally:
        v["train_time_s"] = round(time.time() - t0, 1)
        # Nama kunci di atas adalah placeholder: samakan dengan RUN_FIELDS (kolom yang tak ada di
        # RUN_FIELDS bisa digabung ke 'notes'). Untuk run gagal, kolom metrik diisi None.
        for k in RUN_FIELDS: v.setdefault(k, None)
        log_checked({**{k: v.get(k) for k in RUN_FIELDS}, **{k: v[k] for k in v if k in RUN_FIELDS}})
    return v

RUN_FIELDS: ['run_id', 'date', 'member', 'model', 'purpose', 'ablation_factor', 'factor_value', 'seed', 'train_split', 'val_split', 'imgsz_or_min_size', 'epochs', 'batch_size', 'lr', 'pretrained', 'env', 'config_path', 'commit', 'val_map50', 'val_map50_95', 'val_ap50_platelet', 'val_mae_platelet', 'params_m', 'latency_ms', 'status', 'notes']


In [ ]:
raise RuntimeError("Sel arsip: sudah dijalankan di sesi awal. Jangan dijalankan ulang (lihat catatan di atas).")  # baris ini ditambahkan saat merapikan notebook; output di bawah adalah catatan asli
# (a) Uji asap: 1 epoch, subset kecil. fraction ada di dokumentasi, tapi verifikasi di DEFAULT_CFG-mu
smoke = run_yolo(160, 42, "smoke_y8s_160", epochs=1, note="smoke test, tidak dipakai", fraction=0.05)
print({k: smoke[k] for k in ("status","error","map50","platelet_ap")})
# Verifikasi atribut metrik di versi terpasang (jalankan sekali):
m = YOLO(smoke["best_weights"]).val(data=str(CFG_YAML), split="val", imgsz=160, plots=False)
print([a for a in dir(m.box) if not a.startswith("_")]); print(m.results_dict); print(m.names)

# (b) Baseline
base = run_yolo(640, 42, "y8s_img640_s42", note="baseline")

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/txl_pbc_clean.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.05, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=160, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale

In [ ]:
raise RuntimeError("Sel arsip: sudah dijalankan di sesi awal. Jangan dijalankan ulang (lihat catatan di atas).")  # baris ini ditambahkan saat merapikan notebook; output di bawah adalah catatan asli
import shutil, time, yaml, subprocess
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

MEMBER, BATCH = "anggota2", 16
SRC, DST = Path("/content/runs_yolo"), DRIVE / "runs"
OPT_NOTE = "optimizer: ISI dari baris 'optimizer:' di log training baseline"   # lihat catatan di bawah sel

# 1) Amankan ke Drive DULU
shutil.copytree(SRC, DST, dirs_exist_ok=True)
print(subprocess.run(f"ls {DST}", shell=True, capture_output=True, text=True).stdout)

LOG = Path(f"logs/runs_{MEMBER}.csv")

def eval_and_log(run_name, purpose, ablation_factor="", factor_value="", note="", expect=None):
    rd = SRC / run_name
    a = yaml.safe_load((rd / "args.yaml").read_text())                 # kebenaran dasar: argumen yang BENAR-BENAR dipakai
    best = rd / "weights" / "best.pt"; assert best.exists(), best
    res = pd.read_csv(rd / "results.csv"); res.columns = res.columns.str.strip()
    print(f"[{run_name}] baris results.csv={len(res)} | "
          + str({k: a.get(k) for k in ("data","epochs","patience","batch","imgsz","seed","deterministic","optimizer","lr0","workers","amp","close_mosaic","fraction")}))
    assert Path(a["data"]).resolve() == CFG_YAML, f"data bukan txl_pbc_clean.yaml: {a['data']}"
    if expect:
        for k, v in expect.items(): assert a[k] == v, f"{k}: args={a[k]} != harapan {v}"
        assert len(res) == a["epochs"], "results.csv tidak lengkap: run tidak selesai penuh"
    if LOG.exists() and run_name in LOG.read_text():
        print("sudah ada di log, dilewati:", run_name); return
    ym = YOLO(str(best))
    m = ym.val(data=str(CFG_YAML), split="val", imgsz=a["imgsz"], batch=BATCH, device=0,
               plots=False, project=str(SRC), name=run_name + "_val", exist_ok=True)
    b = m.box
    assert {int(k): str(v) for k, v in m.names.items()} == {0: "WBC", 1: "RBC", 2: "Platelets"}
    pc = {int(c): dict(ap50=float(b.ap50[i]), ap=float(b.ap[i]), p=float(b.p[i]), r=float(b.r[i]))
          for i, c in enumerate(b.ap_class_index)}
    assert 2 in pc, "Platelets tidak ada di hasil val"
    row = {f: None for f in RUN_FIELDS if f != "member"}
    row.update(run_id=run_name, date=time.strftime("%Y-%m-%d"), model="yolov8s", purpose=purpose,
               ablation_factor=ablation_factor, factor_value=factor_value, seed=a["seed"],
               train_split="train_clean", val_split="val_clean", imgsz_or_min_size=a["imgsz"],
               epochs=a["epochs"], batch_size=a["batch"], pretrained="COCO (yolov8s.pt)", env=ENV,
               config_path=str(CFG_YAML.relative_to(Path.cwd())), commit=GIT_COMMIT,
               val_map50=float(b.map50), val_map50_95=float(b.map), val_ap50_platelet=pc[2]["ap50"],
               params_m=round(sum(p.numel() for p in ym.model.parameters()) / 1e6, 2), status="ok",
               notes=" | ".join(x for x in [note, OPT_NOTE,
                     f"platelet AP50-95={pc[2]['ap']:.4f} P={pc[2]['p']:.4f} R={pc[2]['r']:.4f}",
                     f"WBC AP50-95={pc.get(0,{}).get('ap')} RBC AP50-95={pc.get(1,{}).get('ap')}",
                     f"best={DST/run_name/'weights'/'best.pt'}",
                     "dilatih dgn kode lama (log gagal); val dievaluasi ulang dari best.pt",
                     "val_mae_platelet & latency_ms: dari skrip evaluasi bersama"] if x))
    log_run(member=MEMBER, **row)
    print({k: row[k] for k in ("run_id","val_map50","val_map50_95","val_ap50_platelet","params_m")})

eval_and_log("smoke_y8s_160", "smoke", note="smoke test, tidak dipakai")
eval_and_log("y8s_img640_s42", "baseline", ablation_factor="imgsz", factor_value=640,
             note="baseline yolov8s 640", expect=dict(imgsz=640, seed=42, epochs=100, batch=16))

print(pd.read_csv(LOG).iloc[:, [0, 3, 4, 7, 10, 11, 18, 19, 20, 24]].to_string())
shutil.copy(LOG, DRIVE / "logs" / LOG.name)

smoke_y8s_160
smoke_y8s_160_val
y8s_img640_s42
y8s_img640_s42_val

[smoke_y8s_160] baris results.csv=1 | {'data': '/content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/txl_pbc_clean.yaml', 'epochs': 1, 'patience': 1, 'batch': 16, 'imgsz': 160, 'seed': 42, 'deterministic': True, 'optimizer': 'auto', 'lr0': 0.01, 'workers': 2, 'amp': True, 'close_mosaic': 10, 'fraction': 0.05}
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 804.2±331.2 MB/s, size: 32.8 KB)
val: Scanning /content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/raw/txl_pbc/TXL-PBC/labels/val.cache... 239 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 239/239 47.7Mit/s 0.0s
val: /content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/raw/txl_pbc/TXL-PBC/images/val/0ac132c270355979b686c0801e558849.png: 15 duplic

PosixPath('/content/drive/MyDrive/tubes_dl/anggota2/logs/runs_anggota2.csv')

In [ ]:
raise RuntimeError("Sel arsip: sudah dijalankan di sesi awal. Jangan dijalankan ulang (lihat catatan di atas).")  # baris ini ditambahkan saat merapikan notebook; output di bawah adalah catatan asli
r = run_yolo(imgsz, seed, name,
             purpose="ablation" if seed == 42 else "seed_variance",
             ablation_factor="imgsz", factor_value=imgsz,
             note="dimulai ulang setelah sesi terputus di epoch 3" if name == "y8s_img640_s123" else "")

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tugas_besar_deep_learning_a_kelompok_deep_cheating/data/processed/yolo/txl_pbc_clean.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scal